In [1]:
import duckdb

con = duckdb.connect()
con.sql("SET memory_limit='4GB'")

con.sql("""
SELECT COUNT(*) AS nb_produits
FROM 'hf://datasets/openfoodfacts/product-database/food.parquet'
""").show()

┌─────────────┐
│ nb_produits │
│    int64    │
├─────────────┤
│     4882161 │
└─────────────┘



In [2]:
con.sql("""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM 'hf://datasets/openfoodfacts/product-database/food.parquet')
WHERE column_name ILIKE '%countr%'
   OR column_name ILIKE '%brand%'
   OR column_name ILIKE '%categor%'
   OR column_name ILIKE '%nutri%'
   OR column_name IN ('code', 'product_name')
""").df()

,column_name,column_type
0,brands_tags,VARCHAR[]
1,brands,VARCHAR
2,categories,VARCHAR
3,categories_tags,VARCHAR[]
4,categories_properties,"STRUCT(ciqual_food_code INTEGER, agribalyse_fo..."
5,code,VARCHAR
6,compared_to_category,VARCHAR
7,countries_tags,VARCHAR[]
8,main_countries_tags,VARCHAR[]
9,no_nutrition_data,BOOLEAN


In [4]:
import duckdb

con = duckdb.connect()
con.sql("SET memory_limit='4GB'")

con.sql("""
WITH p AS (
    SELECT unnest(countries_tags) AS pays
    FROM 'data/food.parquet'
)
SELECT pays, COUNT(*) AS nb_produits
FROM p
GROUP BY pays
ORDER BY nb_produits DESC
LIMIT 20
""").show(max_rows=25)

con.sql("""
WITH p AS (
    SELECT unnest(countries_tags) AS pays
    FROM 'data/food.parquet'
)
SELECT pays, COUNT(*) AS nb_produits
FROM p
WHERE pays = 'en:morocco'
GROUP BY pays
""").show()

┌───────────────────┬─────────────┐
│       pays        │ nb_produits │
│      varchar      │    int64    │
├───────────────────┼─────────────┤
│ en:france         │     1292692 │
│ en:united-states  │      994452 │
│ en:germany        │      436519 │
│ en:spain          │      362447 │
│ en:italy          │      273558 │
│ en:united-kingdom │      199512 │
│ en:canada         │      129818 │
│ en:netherlands    │      115473 │
│ en:switzerland    │      107263 │
│ en:belgium        │      103553 │
│ en:australia      │       84291 │
│ en:ireland        │       80123 │
│ en:world          │       73290 │
│ en:romania        │       49857 │
│ en:japan          │       47125 │
│ en:poland         │       39471 │
│ en:brazil         │       37691 │
│ en:russia         │       37262 │
│ en:norway         │       29460 │
│ en:sweden         │       28491 │
└───────────────────┴─────────────┘
  20 rows               2 columns

┌────────────┬─────────────┐
│    pays    │ nb_produits │
│  varc